In [2]:
%load_ext autoreload
%autoreload 2
from experiments.policy_sweep import (
    run_replications,
    summarize_ci,
    compare_policies,
    labeled_policy_grid,
)
from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.lowest_soc_diff import LowestSoCDiffQueuePolicy
from policy.queue.closest_power_match import ClosestPowerMatchQueuePolicy
from policy.power.proportional import ProportionalPower
from policy.power.static import StaticPower
import pandas as pd

# To sweep this comparison over a parameter grid instead of one
# scenario, see experiments/run_policy_sweep.py and
# policy_sweep_results.ipynb -- same machinery, recorded to disk.


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# --- inputs ---
# mx_wait is filled from the first FIFO run, then applied to other queue rules.
mx_wait = None

queue_policies = [
    FIFOQueuePolicy(),
    LowestSoCDiffQueuePolicy(),
]
#    ClosestPowerMatchQueuePolicy(),
#]
queue_names = ["FIFO", "LSoCD"] #, , "L_SoC_D", "P_Match"]

# Parallel list: one name per power-policy instance (same idea as queue_names).
# One power policy → scenario labels stay queue names (FIFO, L_SoC_D, ...).
# Several power policies and several queue policies → FIFO_Prop, FIFO_Static, ...
power_policies = [
    #StaticPower(),
    ProportionalPower()
    # StaticPower(),  # equal module split; leftovers by unmet p_req
]
power_names = ["Prop"]
#power_names = ["Prop", "Static"]  # with both: FIFO_Prop, FIFO_Static, ...

scenario = {
    "n_piles": 1,
    "n_connectors": 2,
    "n_modules": 6,
    "p_module": 25,
    "queue_capacity": 100,
    "mean_interarrival": 15,
    "delta_arr": None,  # None: continuous; 1: nearest minute
    # None -> config.WARMUP_PERIOD (0.0 out of the box, no warm-up). See
    # sim_benchmark.ipynb for the full usage example. run_replications
    # below always reports BOTH windows regardless (each metric plus its
    # "{name} (measured)" post-warm-up counterpart, from the same episode
    # at no extra cost) -- they're just identical whenever warmup_period=0.
    "warmup_period": None,
    "flush_queue_at_warmup": False,
}

n_reps = 30
confidence = 0.95
seed0 = 0

grid = labeled_policy_grid(
    queue_policies, queue_names, power_policies, power_names
)

# --- run ---
reps = {}
summaries = {}
mean_cols = {}

for label, q_pol, p_pol, q_name, _p_name in grid:
    if q_name != "FIFO" and mx_wait:
        q_pol.max_wait = mx_wait * 0.75
    r = run_replications(
        scenario={**scenario, "queue_policy": q_pol, "power_policy": p_pol},
        n_reps=n_reps,
        seed0=seed0,
    )
    s = summarize_ci(r, confidence=confidence)
    if q_name == "FIFO" and mx_wait is None:
        mx_wait = s.loc["max wait time", "mean"]

    reps[label] = r
    summaries[label] = s
    mean_cols[f"{label}_mean"] = s["mean"]

# paired CRN comparisons (same seeds)
signs, pair_details = compare_policies(reps, confidence=confidence)


In [8]:
reps['FIFO']

,seed,finished EVs,finished EVs (measured),dropped EVs,dropped EVs (measured),total energy delivered (kWh),total energy delivered (kWh) (measured),average L,average L (measured),average Q,...,max wait time,max wait time (measured),avg charge time,avg charge time (measured),avg sys time,avg sys time (measured),util_rho_sim,util_rho_sim (measured),util_rho_theory,util_rho_theory (measured)
0,1.0,34.0,34.0,0.0,0.0,1606.790243,1606.790243,4.620866,4.620866,2.786763,...,136.464055,136.464055,37.638943,37.638943,95.184342,95.184342,0.917051,0.917051,0.888697,0.888697
1,2.0,36.0,36.0,0.0,0.0,1744.813190,1744.813190,9.146368,9.146368,7.156337,...,191.011602,191.011602,38.152091,38.152091,136.669136,136.669136,0.995016,0.995016,0.953802,0.953802
2,3.0,33.0,33.0,0.0,0.0,1680.181199,1680.181199,10.833829,10.833829,8.905241,...,269.214548,269.214548,40.441344,40.441344,168.203422,168.203422,0.964294,0.964294,0.926781,0.926781
3,4.0,29.0,29.0,0.0,0.0,1516.003938,1516.003938,3.749580,3.749580,1.980201,...,94.115149,94.115149,42.759688,42.759688,75.896283,75.896283,0.884689,0.884689,0.861133,0.861133
4,5.0,33.0,33.0,0.0,0.0,1635.114598,1635.114598,14.294694,14.294694,12.398992,...,266.716302,266.716302,40.034579,40.034579,197.207868,197.207868,0.947851,0.947851,0.917459,0.917459
5,6.0,35.0,35.0,0.0,0.0,1710.832521,1710.832521,11.806544,11.806544,9.839169,...,261.616505,261.616505,39.304357,39.304357,176.948267,176.948267,0.983688,0.983688,0.955314,0.955314
6,7.0,35.0,35.0,0.0,0.0,1663.928039,1663.928039,7.314969,7.314969,5.407815,...,209.816662,209.816662,38.088445,38.088445,108.577924,108.577924,0.953577,0.953577,0.925761,0.925761
7,8.0,38.0,38.0,0.0,0.0,1655.244597,1655.244597,11.848653,11.848653,9.945409,...,251.284272,251.284272,34.603980,34.603980,167.128597,167.128597,0.951622,0.951622,0.913161,0.913161
8,9.0,34.0,34.0,0.0,0.0,1564.595593,1564.595593,4.805534,4.805534,2.984299,...,87.949981,87.949981,37.184012,37.184012,73.344088,73.344088,0.910618,0.910618,0.877956,0.877956
9,10.0,34.0,34.0,0.0,0.0,1693.623516,1693.623516,11.306915,11.306915,9.390522,...,244.805451,244.805451,39.163530,39.163530,145.743549,145.743549,0.958197,0.958197,0.924694,0.924694


In [22]:
signs  # -1: first policy lower; 0: n.s.; 1: second policy lower


,FIFO|LSoCD
metric,
finished EVs,-1
dropped EVs,0
total energy delivered (kWh),0
average L,1
average Q,1
avg wait time,1
max wait time,-1
avg charge time,1
avg sys time,1
